# IQB-Edu
### Moldova School Connectivity — January 2026

> **How to use this notebook**
> Press **Run → Run All Cells** (or Shift+Enter through each cell).
> The interactive panel will appear at the bottom of the page.
> No coding needed — everything is controlled by sliders and toggles in this notebook.

---

## What question does this tool answer?

_Given what we want schools to be able to do online, how many schools in Moldova can actually do it?_

Internet connectivity is not equally valuable for every educational goal.
A school that can comfortably support web browsing and audio may still struggle with
live video conferencing. A connection that looks adequate on a typical day may fail
students during peak hours.

The **Internet Quality Barometer for Education (IQB-Edu)** translates raw network
measurements — download speed, upload speed, latency, and packet loss — into a
single score (0–1) that reflects how well a school's connection supports a given
set of educational activities.

This tool puts the policy choices in your hands:

| Control | What it does |
|---|---|
| **Use Case Standards** | Per activity: the minimum Download, Upload, Latency and Packet Loss it requires, and how much it weighs (Weighting) in the composite score |
| **Performance Standard** | Whether to evaluate typical days, worst-case days, or peak capacity |
| **IQB-Edu Scores for Schools** | Pick a threshold to see how many schools perform above it |

The charts update the moment you change any setting, showing you exactly how many
schools clear the bar — and which ones don't.

---


In [13]:
# ── Imports ─────────────────────────────────────────────────────────────────
import warnings, copy
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from iqbedu.data import IQBEduData
from iqb.calculator import IQBCalculator, IQB_CONFIG


In [14]:
# ── Data loading ────────────────────────────────────────────────────────────
STATS_DATA_PATH    = "./data/IQB_Edu_MD_2026-01-01_2026-02-01_stats.csv"
GEO_DATA_PATH      = "./data/giga_MD_schools_metadata.csv"
COUNTRY_STATS_PATH = "./data/M-Lab_MD_2026-01-01_2026-02-01_country_stats.csv"

df_stats   = pd.read_csv(STATS_DATA_PATH)
df_geo     = pd.read_csv(GEO_DATA_PATH)
df_country = pd.read_csv(COUNTRY_STATS_PATH)
data       = IQBEduData(df=df_stats)
IQB_USE_CASES = list(IQB_CONFIG["use cases"].keys())

print(f"✓  {len(data.schools)} schools loaded  |  "
      f"{int(df_stats['download_sample_count'].sum()):,} M-Lab measurements  |  "
      f"Period: January 2026")


✓  57 schools loaded  |  3,509 M-Lab measurements  |  Period: January 2026


## 🏫 About This School Sample

The 57 schools in this dataset are Moldova's **IQB-Edu "model schools"** — a small,
purposefully chosen set of sites, not a random or representative national sample.
They were selected to be **geographically dispersed** across the country, and each
received a specific connectivity intervention: a **GigaMeter** device installed
on site to continuously run M-Lab NDT speed tests (4 tests/day), which is what
produced the measurements used throughout this notebook.

In [15]:
# ── Defaults ───────────────────────────────────────────────────────────────
# Starting point for each use case's Weighting slider (its share of the composite score).
DEFAULT_WEIGHTS = {
    "web browsing": 1.0, "audio streaming": 0.5, "video streaming": 1.0,
    "video conferencing": 1.0, "online backup": 0.5, "gaming": 0.0,
}

UC_LABELS = {
    "web browsing":       "🌐  Web Browsing",
    "audio streaming":    "🎵  Audio Streaming",
    "video streaming":    "📺  Video Streaming",
    "video conferencing": "📹  Video Conferencing",
    "online backup":      "☁️   Cloud Storage / Backup",
    "gaming":             "🎮  Educational Gaming",
}

PERC_OPTIONS = {
    "Connection capacity  (near-best — p95)":                   95,
    "Typical / average experience  (median — p50)":             50,
    "Conservative / difficult day experience  (p25)":           25,
}

# Default network-requirement thresholds per use case, read straight from the
# library config so the sliders start where the IQB defaults do.
DEFAULT_THRESHOLDS = {
    uc: {
        "download":    IQB_CONFIG["use cases"][uc]["network requirements"]["download_throughput_mbps"]["threshold min"],
        "upload":      IQB_CONFIG["use cases"][uc]["network requirements"]["upload_throughput_mbps"]["threshold min"],
        "latency":     IQB_CONFIG["use cases"][uc]["network requirements"]["latency_ms"]["threshold min"],
        "packet_loss": IQB_CONFIG["use cases"][uc]["network requirements"]["packet_loss"]["threshold min"],
    }
    for uc in IQB_USE_CASES
}

# ── Score computation with caching ────────────────────────────────────────────
_score_cache = {}

def _apply_uc_settings(cfg, uc, settings):
    cfg["use cases"][uc]["w"] = settings["weight"]
    nr = cfg["use cases"][uc]["network requirements"]
    nr["download_throughput_mbps"]["threshold min"] = settings["download"]
    nr["upload_throughput_mbps"]["threshold min"]   = settings["upload"]
    nr["latency_ms"]["threshold min"]               = settings["latency"]
    nr["packet_loss"]["threshold min"]              = settings["packet_loss"]

def compute_scores(uc_settings, percentile):
    key = (
        tuple(
            (uc, s["weight"], s["download"], s["upload"], s["latency"], s["packet_loss"])
            for uc, s in sorted(uc_settings.items())
        ),
        percentile,
    )
    if key in _score_cache:
        return _score_cache[key]

    cfg = copy.deepcopy(IQB_CONFIG)
    for uc in IQB_USE_CASES:
        _apply_uc_settings(cfg, uc, uc_settings[uc])
    calc = IQBCalculator(config=cfg)

    rows = []
    for sid in data.schools:
        mlab = data.to_iqb_data(sid, percentile=percentile)
        rows.append({"school_id": sid,
                     "score": calc.calculate_iqb_score({"m-lab": mlab.to_dict()})})

    df = pd.DataFrame(rows)
    df = pd.merge(df, data.df[["school_id", "giga_id_school",
                                "download_sample_count"]], on="school_id", how="left")
    df = pd.merge(df, df_geo, on="giga_id_school", how="left")
    _score_cache[key] = df
    return df

_uc_score_cache = {}

def compute_uc_scores(uc, settings, percentile):
    key = (uc, settings["download"], settings["upload"], settings["latency"],
           settings["packet_loss"], percentile)
    if key in _uc_score_cache:
        return _uc_score_cache[key]
    cfg = copy.deepcopy(IQB_CONFIG)
    for u in IQB_USE_CASES:
        cfg["use cases"][u]["w"] = 1 if u == uc else 0
    _apply_uc_settings(cfg, uc, {**settings, "weight": 1})
    calc = IQBCalculator(config=cfg)
    scores = [
        calc.calculate_iqb_score(
            {"m-lab": data.to_iqb_data(sid, percentile=percentile).to_dict()}
        )
        for sid in data.schools
    ]
    _uc_score_cache[key] = scores
    return scores

print("✓  Computation helpers ready")


✓  Computation helpers ready


---
## 🎛️  Use Case Explorer

The IQB score is calculated based on 6 use cases. Each use case has a set of network requirements: if the network metrics are above the selected threshold (e.g., Download throughput > 10 Mbps), then we assess that Internet quality is good enough for this use case.

> We ask you to help us refine the design of IQB-Edu by selecting thresholds for the network requirements of each use case. Below, you can experiment with different threshold values.

**Network requirements (thresholds) per use case:** Expand a use case below to set its requirements, then observe how school readiness changes. The charts and summary table update automatically.
_Tip: check the "Network Metric Distributions" section below to help you choose thresholds to experiment with._

**Use case weights:** Each use case contributes to the IQB score according to its assigned weight. For IQB-Edu, we selected default weights based on each use case's relevance to education, shown below. _We recommend keeping the default weights._

**Performance Standard:** Threshold comparisons are made against measured network performance. Since measurements vary from day to day, we aggregate each school's results into a distribution and compare thresholds against a chosen percentile of it. For example, selecting the median (50th percentile) means the IQB score represents a school's "typical/average experience," while selecting the 95th percentile means it represents the school's "connection capacity" (best-case performance).

In [16]:
# ═══════════════════════════════════════════════════════════════════════════
#  Run this cell to launch the interactive explorer.
# ═══════════════════════════════════════════════════════════════════════════
style  = {"description_width": "240px"}
layout = widgets.Layout(width="580px")

# ── Widgets ──────────────────────────────────────────────────────────────
# Weighting = each use case's share of the composite score (0 = ignored, 1 = top priority).
# Always visible (not tucked inside the accordion) so they're easy to compare at a glance.
weight_sliders = {
    uc: widgets.FloatSlider(
        value=DEFAULT_WEIGHTS[uc],
        min=0.0, max=1.0, step=0.1,
        description=UC_LABELS[uc],
        style=style, layout=layout,
        readout_format=".1f",
    )
    for uc in IQB_USE_CASES
}

# Per-use-case minimum network requirements — the actual pass/fail bar for that activity.
threshold_sliders = {
    uc: {
        "download": widgets.FloatSlider(
            value=DEFAULT_THRESHOLDS[uc]["download"], min=0, max=100, step=5,
            description="⬇️ Download min (Mbps)", style=style, layout=layout,
            readout_format=".0f",
        ),
        "upload": widgets.FloatSlider(
            value=DEFAULT_THRESHOLDS[uc]["upload"], min=0, max=50, step=5,
            description="⬆️ Upload min (Mbps)", style=style, layout=layout,
            readout_format=".0f",
        ),
        "latency": widgets.FloatSlider(
            value=DEFAULT_THRESHOLDS[uc]["latency"], min=0, max=200, step=5,
            description="⏱️ Latency max (ms)", style=style, layout=layout,
            readout_format=".0f",
        ),
        "packet_loss": widgets.FloatSlider(
            value=DEFAULT_THRESHOLDS[uc]["packet_loss"], min=0.0, max=0.05, step=0.001,
            description="📉 Packet loss max", style=style, layout=layout,
            readout_format=".1%",
        ),
    }
    for uc in IQB_USE_CASES
}

uc_accordion = widgets.Accordion(children=[
    widgets.VBox([
        threshold_sliders[uc]["download"],
        threshold_sliders[uc]["upload"],
        threshold_sliders[uc]["latency"],
        threshold_sliders[uc]["packet_loss"],
    ])
    for uc in IQB_USE_CASES
])
for i, uc in enumerate(IQB_USE_CASES):
    uc_accordion.set_title(i, UC_LABELS[uc])
uc_accordion.selected_index = None

perc_toggle = widgets.RadioButtons(
    options=list(PERC_OPTIONS.keys()),
    value="Typical / average experience  (median — p50)",
    description="Evaluate schools on:",
    style=style,
)

threshold_sl = widgets.FloatSlider(
    value=0.6, min=0.0, max=1.0, step=0.05,
    description="Threshold:",
    style=style, layout=layout,
    readout_format=".2f",
)

output = widgets.Output()

# ── Score-to-words helper ─────────────────────────────────────────────────
def score_label(s):
    if s >= 0.85: return "Ready"
    if s >= 0.65: return "Mostly ready"
    if s >= 0.45: return "Partially ready"
    return "Not ready"

def score_color(s, threshold):
    if s >= threshold:   return "#2ca02c"
    if s >= threshold * 0.75: return "#ff7f0e"
    return "#d62728"

# ── Main refresh ──────────────────────────────────────────────────────────
def refresh(change):
    uc_settings = {
        uc: {
            "weight":      weight_sliders[uc].value,
            "download":    threshold_sliders[uc]["download"].value,
            "upload":      threshold_sliders[uc]["upload"].value,
            "latency":     threshold_sliders[uc]["latency"].value,
            "packet_loss": threshold_sliders[uc]["packet_loss"].value,
        }
        for uc in IQB_USE_CASES
    }
    percentile = PERC_OPTIONS[perc_toggle.value]
    threshold  = threshold_sl.value
    perc_short = perc_toggle.value.split("(")[1].rstrip(")")

    df       = compute_scores(uc_settings, percentile)
    pct_ok   = (df["score"] >= threshold).mean() * 100
    pct_fail = 100 - pct_ok
    median_s = df["score"].median()

    df_sorted = df.sort_values("score").reset_index(drop=True)

    # ── Build figure ─────────────────────────────────────────────────────
    fig = make_subplots(
        rows=1, cols=3,
        column_widths=[0.27, 0.38, 0.35],
        subplot_titles=[
            "Weighting Profile",
            "Score Distribution — all schools",
            "Schools Ranked by Score",
        ],
        specs=[[{"type": "polar"}, {"type": "xy"}, {"type": "xy"}]],
        horizontal_spacing=0.09,
    )

    # Radar — policy shape
    ucs   = IQB_USE_CASES
    r_v   = [uc_settings[u]["weight"] for u in ucs] + [uc_settings[ucs[0]]["weight"]]
    theta = [UC_LABELS[u] for u in ucs] + [UC_LABELS[ucs[0]]]
    fig.add_trace(go.Scatterpolar(
        r=r_v, theta=theta, fill="toself",
        fillcolor="rgba(33,102,172,0.20)",
        line=dict(color="rgb(33,102,172)", width=2),
        name="Weighting",
    ), row=1, col=1)
    fig.update_polars(
        radialaxis=dict(range=[0, 1], showticklabels=False, ticks=""),
        angularaxis=dict(tickfont=dict(size=10)),
    )

    # Histogram
    fig.add_trace(go.Histogram(
        x=df["score"], nbinsx=16,
        marker_color="rgba(33,102,172,0.55)",
        name="Schools",
    ), row=1, col=2)
    # add_vline/add_hline with row/col trips on the Scatterpolar trace;
    # use add_shape with explicit axis references instead.
    # col 2 (histogram) → xref="x", yref="y domain"
    # col 3 (bar)       → xref="x2 domain", yref="y2"
    fig.add_shape(type="line", x0=threshold, x1=threshold, y0=0, y1=1,
                  xref="x", yref="y domain",
                  line=dict(dash="dash", color="crimson", width=1.5))
    fig.add_annotation(x=threshold, y=0.97, xref="x", yref="y domain",
                       text=f"min {threshold:.2f}", font=dict(color="crimson", size=10),
                       showarrow=False, yanchor="top", xanchor="left")
    fig.add_shape(type="line", x0=median_s, x1=median_s, y0=0, y1=1,
                  xref="x", yref="y domain",
                  line=dict(dash="dot", color="steelblue", width=1.5))
    fig.add_annotation(x=median_s, y=0.82, xref="x", yref="y domain",
                       text=f"median {median_s:.2f}", font=dict(color="steelblue", size=10),
                       showarrow=False, yanchor="top", xanchor="left")
    fig.update_xaxes(range=[0, 1], title_text="IQB Score", row=1, col=2)
    fig.update_yaxes(title_text="# Schools", row=1, col=2)

    # Bar — ranked schools
    bar_colors = [score_color(s, threshold) for s in df_sorted["score"]]
    fig.add_trace(go.Bar(
        x=list(range(len(df_sorted))),
        y=df_sorted["score"],
        marker_color=bar_colors,
        hovertext=df_sorted["school_id"],
        hovertemplate=(
            "<b>School:</b> %{hovertext}<br>"
            "<b>Score:</b> %{y:.3f}<extra></extra>"
        ),
        name="Score",
    ), row=1, col=3)
    fig.add_shape(type="line", x0=0, x1=1, y0=threshold, y1=threshold,
                  xref="x2 domain", yref="y2",
                  line=dict(dash="dash", color="crimson", width=1.5))
    fig.update_xaxes(
        title_text="Schools (low → high score)", showticklabels=False, row=1, col=3,
    )
    fig.update_yaxes(range=[0, 1.05], title_text="IQB Score", row=1, col=3)

    n_ok   = int(round(pct_ok / 100 * len(df)))
    n_fail = len(df) - n_ok
    title  = (
        f"{perc_short}  ·  "
        f"<span style='color:#2ca02c'>{n_ok} schools ready</span>  "
        f"<span style='color:#d62728'>/ {n_fail} not ready</span>  "
        f"(threshold {threshold:.2f},  median {median_s:.2f})"
    )
    fig.update_layout(
        height=430,
        showlegend=False,
        margin=dict(t=70, b=40, l=40, r=20),
        title_text=title,
        title_font_size=12,
    )

    # ── Per-use-case breakdown table ─────────────────────────────────────
    rows = []
    for uc in IQB_USE_CASES:
        w = uc_settings[uc]["weight"]
        if w == 0:
            continue
        uc_scores = compute_uc_scores(uc, uc_settings[uc], percentile)
        med   = float(np.median(uc_scores))
        p_ok  = float(np.mean([s >= 0.5 for s in uc_scores])) * 100
        p_thr = float(np.mean([s >= threshold for s in uc_scores])) * 100
        rows.append({
            "Use Case":              UC_LABELS[uc],
            "Weighting":             "●" * int(round(w * 5)) + "○" * (5 - int(round(w * 5))),
            "Median School Score":   f"{med:.2f}  ({score_label(med)})",
            "% Schools ≥ 0.50":     f"{p_ok:.0f}%",
            f"% Schools ≥ {threshold:.2f}": f"{p_thr:.0f}%",
        })

    with output:
        clear_output(wait=True)
        fig.show()

        if rows:
            tbl_html = (
                pd.DataFrame(rows)
                .style
                .set_properties(**{"text-align": "center", "padding": "4px 12px"})
                .set_table_styles([
                    {"selector": "th",
                     "props": [("background", "#f0f4f8"), ("text-align", "center"),
                               ("padding", "6px 12px")]},
                ])
                .hide(axis="index")
                .to_html()
            )
            display(HTML(
                "<h4 style='margin-top:18px'>Use Case Breakdown "
                "<span style='font-weight:normal;font-size:0.85em;color:#666'>"
                "(active priorities only)</span></h4>"
                + tbl_html
            ))

        # Plain-language summary
        gap = threshold - median_s
        if gap > 0:
            gap_msg = (
                f"The median school score ({median_s:.2f}) is "
                f"<b>{gap:.2f} points below</b> your threshold. "
                "Improving connectivity across the system would close this gap."
            )
        else:
            gap_msg = (
                f"The median school score ({median_s:.2f}) already "
                f"<b>exceeds</b> your threshold — most schools are on track."
            )
        display(HTML(
            f"<p style='margin-top:12px;color:#333;font-size:0.93em'>"
            f"<b>Reading the results:</b> Under your current settings, "
            f"<b>{pct_ok:.0f}%</b> of schools ({n_ok} of {len(df)}) meet your minimum score. "
            f"{gap_msg}</p>"
        ))

# ── Wire observers ────────────────────────────────────────────────────────
for sl in weight_sliders.values():
    sl.observe(refresh, names="value")
for th in threshold_sliders.values():
    for sl in th.values():
        sl.observe(refresh, names="value")
perc_toggle.observe(refresh, names="value")
threshold_sl.observe(refresh, names="value")

# ── Layout ────────────────────────────────────────────────────────────────
sep = "<hr style='margin:10px 0;border-color:#ddd'>"
controls = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 2px'>📶 Use Case Standards</h3>"
                 "<p style='margin:0 0 8px;font-size:0.87em;color:#555'>"
                 "Expand an activity to set its minimum Download, Upload, Latency and "
                 "Packet Loss requirements.</p>"),
    uc_accordion,

    widgets.HTML(sep + "<h3 style='margin:0 0 2px'>📅 Performance Standard</h3>"
                 "<p style='margin:0 0 8px;font-size:0.87em;color:#555'>"
                 "Which slice of the month's measurements to evaluate</p>"),
    perc_toggle,

    widgets.HTML(sep + "<h3 style='margin:0 0 2px'>⚖️ Weighting</h3>"
                 "<p style='margin:0 0 8px;font-size:0.87em;color:#555'>"
                 "How much each activity counts toward the composite score "
                 "(0 = not a priority · 1 = top priority).</p>"),
    *[weight_sliders[uc] for uc in IQB_USE_CASES],

    widgets.HTML(sep + "<h3 style='margin:0 0 2px'>📊 IQB-Edu Scores for Schools</h3>"
                 "<p style='margin:0 0 8px;font-size:0.87em;color:#555'>"
                 "Pick a threshold to see how many schools perform above it. "
                 "Schools below the threshold appear red in the ranked chart.</p>"),
    threshold_sl,
])

display(widgets.VBox([controls, output]))
refresh(None)


---
## 📊 Network Metric Distributions

The IQB score compresses each school down to a single number. Before you settle on
threshold values above, it helps to see the **raw measurements** those thresholds
are drawn against:

- **Across all schools** — how a metric (e.g. latency) is spread across the 57
  schools at a chosen percentile.
- **Single school** — how *that one school's* measurements vary from a hard day
  (p1) to its best day (p99).
- **Country statistics** — the cumulative distribution (CDF) of *all* M-Lab
  measurements collected across Moldova over the period (i.e. not only from schools).  

In [17]:
# ── Network metric distribution explorer ──────────────────────────────────────
from iqbedu.data import VALID_PERCENTILES

METRIC_OPTIONS = {
    "⬇️ Download (Mbps)": "download",
    "⬆️ Upload (Mbps)":   "upload",
    "⏱️ Latency (ms)":     "latency",
    "📉 Packet Loss":      "loss",
}

# Independent Layout instances — must NOT reuse the `layout` object shared by the
# Use Case Standards sliders above: toggling .display on a shared Layout hides
# every widget that references it, not just this one.
metric_dd = widgets.Dropdown(
    options=list(METRIC_OPTIONS.keys()), value="⬇️ Download (Mbps)",
    description="Metric:", style=style, layout=widgets.Layout(width="580px"),
)

dist_view_toggle = widgets.RadioButtons(
    options=["Across all schools", "Single school", "Country statistics"],
    value="Across all schools",
    description="View:", style=style,
)

school_dd = widgets.Dropdown(
    options=data.schools, description="School:", style=style,
    layout=widgets.Layout(width="580px"),
)

dist_perc_toggle = widgets.RadioButtons(
    options=list(PERC_OPTIONS.keys()),
    value="Typical / average experience  (median — p50)",
    description="Evaluate at:", style=style,
)

dist_output = widgets.Output()

def refresh_dist(change):
    prefix = METRIC_OPTIONS[metric_dd.value]

    with dist_output:
        clear_output(wait=True)

        if dist_view_toggle.value == "Across all schools":
            percentile = PERC_OPTIONS[dist_perc_toggle.value]
            perc_short = dist_perc_toggle.value.split("(")[1].rstrip(")")
            col = f"{prefix}_p{percentile}"
            fig = go.Figure(go.Histogram(
                x=df_stats[col], nbinsx=16,
                marker_color="rgba(33,102,172,0.55)",
            ))
            fig.update_layout(
                title=f"{metric_dd.value} — distribution across {len(df_stats)} "
                      f"schools ({perc_short})",
                xaxis_title=metric_dd.value, yaxis_title="# Schools",
                height=360, margin=dict(t=60, b=40, l=40, r=20),
            )
        elif dist_view_toggle.value == "Single school":
            row = df_stats[df_stats["school_id"] == school_dd.value].iloc[0]
            xs = [f"p{p}" for p in VALID_PERCENTILES]
            ys = [row[f"{prefix}_p{p}"] for p in VALID_PERCENTILES]
            fig = go.Figure(go.Bar(x=xs, y=ys, marker_color="rgba(33,102,172,0.55)"))
            fig.update_layout(
                title=f"{metric_dd.value} — {school_dd.value} "
                      f"(within-school spread, hardest → best day)",
                xaxis_title="Percentile", yaxis_title=metric_dd.value,
                height=360, margin=dict(t=60, b=40, l=40, r=20),
            )
        else:  # Country statistics
            row = df_country.iloc[0]
            xs = [row[f"{prefix}_p{p}"] for p in VALID_PERCENTILES]
            ys = [p / 100 for p in VALID_PERCENTILES]
            fig = go.Figure(go.Scatter(
                x=xs, y=ys, mode="lines+markers",
                line=dict(color="rgba(33,102,172,0.85)"),
                marker=dict(size=6),
            ))
            fig.update_layout(
                title=f"{metric_dd.value} — cumulative distribution, "
                      f"all Moldova measurements",
                xaxis_title=metric_dd.value, yaxis_title="Cumulative probability",
                yaxis=dict(range=[0, 1], tickformat=".0%"),
                height=360, margin=dict(t=60, b=40, l=40, r=20),
            )
        fig.show()

def _toggle_dist_controls(change):
    view = dist_view_toggle.value
    school_dd.layout.display        = "" if view == "Single school" else "none"
    dist_perc_toggle.layout.display = "" if view == "Across all schools" else "none"
    refresh_dist(None)

metric_dd.observe(refresh_dist, names="value")
school_dd.observe(refresh_dist, names="value")
dist_perc_toggle.observe(refresh_dist, names="value")
dist_view_toggle.observe(_toggle_dist_controls, names="value")
_toggle_dist_controls(None)

display(widgets.VBox([
    widgets.HBox([metric_dd, dist_view_toggle]),
    school_dd,
    dist_perc_toggle,
    dist_output,
]))
refresh_dist(None)


---
## 🔍 Methodology & Transparency

The IQB-Edu framework converts raw network measurements into educational readiness scores.
The sections below explain each step — expand them to inspect the details.

<details>
<summary><b>How are IQB scores calculated?</b></summary>

### Step 1 — Measurements
M-Lab's NDT test runs from each school's network and records:
- **Download throughput** (Mbps)
- **Upload throughput** (Mbps)
- **Round-trip latency** (ms)
- **Packet loss** (fraction 0–1)

Moldova schools in this dataset were measured throughout January 2026.
Each school has a distribution of results, not a single number.

### Step 2 — Percentiles
Instead of using a single measurement, we pick a **percentile** of the distribution:

| Percentile | What it represents |
|---|---|
| p25 | Conditions during the worst quarter of measurements — a hard day |
| p50 | The median — a typical day |
| p95 | Near-peak — what the connection can do at its best |

Choosing p25 is the most demanding test; p95 the most generous.

### Step 3 — Per-use-case scoring
For each use case (web browsing, video conferencing, etc.) the IQB defines
**minimum network requirements**. A school's score for that use case is 1 if all
requirements are met, 0 if none are, and scales smoothly between those extremes
based on how close each metric comes to its threshold.

### Step 4 — Weighted composite
The final IQB score is a **weighted average** of per-use-case scores.
The weights you set above control how much each use case contributes to the composite.
Setting a weight to 0 removes that use case entirely.

### Caveats
- Schools with fewer than ~30 measurements have noisier scores; treat them as indicative.
- NDT tests reflect actual usage conditions (congestion, ISP throttling), not raw capacity.
- IQB thresholds represent current best-practice standards and can be updated as norms evolve.



</details>